# 📘 Module 4.3 – Shape Detection

📌 Goal: Identify geometric shapes from contours


🧠 CORE IDEA (REMEMBER THIS)

Shapes are detected by approximating contours into fewer points

### 🔹 STEP 0 – BASE SETUP (STANDARD PIPELINE)

In [2]:
import cv2
import numpy as np

# Load image
Image = cv2.imread("image.jpg")
img = cv2.resize(Image, (450, 300))
if img is None:
    print("Image not found")
    exit()

# Preprocessing
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5, 5), 0)

_, binary = cv2.threshold(
    blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

# Find contours
contours, _ = cv2.findContours(
    binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
)


### ✅ 1️⃣ SHAPE APPROXIMATION (KEY CONCEPT 🔥)

- 📌 Reduces contour points
- 📌 Makes shape recognition possible

Example – Approximate Contour

In [4]:
output = img.copy()

for cnt in contours:
    # Approximation accuracy (2% of perimeter)
    epsilon = 0.02 * cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, epsilon, True)

    cv2.drawContours(output, [approx], -1, (0, 255, 0), 2)

cv2.imshow("Approximated Shapes", output)
cv2.waitKey(0)
cv2.destroyAllWindows()


### ✅ 2️⃣ DETECT SQUARES & RECTANGLES

📌 Squares & rectangles → 4 vertices

In [5]:
shape_img = img.copy()

for cnt in contours:
    epsilon = 0.02 * cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, epsilon, True)

    if len(approx) == 4:
        x, y, w, h = cv2.boundingRect(approx)
        aspect_ratio = float(w) / h

        if 0.95 <= aspect_ratio <= 1.05:
            shape = "Square"
        else:
            shape = "Rectangle"

        cv2.drawContours(shape_img, [approx], -1, (255, 0, 0), 2)
        cv2.putText(
            shape_img, shape, (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2
        )

cv2.imshow("Squares & Rectangles", shape_img)
cv2.waitKey(0)
cv2.destroyAllWindows()


### ✅ 3️⃣ DETECT CIRCLES

- 📌 Circles → many vertices
- 📌 Use circularity

#### Circularity Formula
Circularity = 4π × Area / (Perimeter²)


📌 Circle ≈ 1.0

In [6]:
circle_img = img.copy()

for cnt in contours:
    area = cv2.contourArea(cnt)
    perimeter = cv2.arcLength(cnt, True)

    if perimeter == 0:
        continue

    circularity = 4 * np.pi * area / (perimeter * perimeter)

    if circularity > 0.8:
        cv2.drawContours(circle_img, [cnt], -1, (0, 255, 0), 2)

        (x, y), radius = cv2.minEnclosingCircle(cnt)
        cv2.circle(circle_img, (int(x), int(y)), int(radius), (0, 0, 255), 2)

        cv2.putText(
            circle_img, "Circle", (int(x - radius), int(y - radius)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2
        )

cv2.imshow("Detected Circles", circle_img)
cv2.waitKey(0)
cv2.destroyAllWindows()


# 🔍 FINAL SHAPE DETECTOR (ALL TOGETHER)

In [7]:
final = img.copy()

for cnt in contours:
    epsilon = 0.02 * cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, epsilon, True)
    area = cv2.contourArea(cnt)

    if area < 500:
        continue

    shape = "Unknown"

    if len(approx) == 3:
        shape = "Triangle"
    elif len(approx) == 4:
        x, y, w, h = cv2.boundingRect(approx)
        aspect_ratio = float(w) / h
        shape = "Square" if 0.95 <= aspect_ratio <= 1.05 else "Rectangle"
    elif len(approx) > 6:
        shape = "Circle"

    cv2.drawContours(final, [approx], -1, (0, 255, 0), 2)

    M = cv2.moments(cnt)
    if M["m00"] != 0:
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
        cv2.putText(
            final, shape, (cx - 20, cy),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2
        )

cv2.imshow("Final Shape Detection", final)
cv2.waitKey(0)
cv2.destroyAllWindows()


### 🧠 SHAPE RULES (REMEMBER ⭐)
Shape	-------------- Condition
- Triangle	---------- 3 vertices
- Square	---------- 4 vertices + aspect ≈ 1
- Rectangle	---------- 4 vertices
- Circle	---------- Many vertices / high circularity

-----
# MINI PROJECT
- 📌 Shape Detector + Object Counter
- 📌 SINGLE PYTHON FILE
- 📌 Example-first, heavily commented, beginner → expert friendly


🎯 WHAT THIS MINI PROJECT DOES

- ✔ Detects multiple objects
- ✔ Identifies shapes (Triangle, Square, Rectangle, Circle)
- ✔ Counts total objects
- ✔ Labels each object on the image


📌 This is real-world logic used in:

- Industrial inspection
- Robotics vision
- Traffic sign analysis
- Object counting systems

### 🧠 COMPLETE PIPELINE (IMPORTANT)
Image
 - Grayscale
 - Blur
 - Threshold
 - Morphology
 - Contours
 - Shape Approximation
 - Shape Classification
 - Object Count

In [9]:
import cv2
import numpy as np

# ================================
# STEP 1: LOAD IMAGE
# ================================

Image = cv2.imread("image.jpg")
img = cv2.resize(Image, (600, 400))
if img is None:
    print("Image not found!")
    exit()

cv2.imshow("Original Image", img)
cv2.waitKey(0)

# ================================
# STEP 2: PREPROCESSING
# ================================

# Convert to grayscale
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Blur to remove noise
blur = cv2.GaussianBlur(gray, (5, 5), 0)

# Convert to binary image
_, binary = cv2.threshold(
    blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

# Improve object boundaries
kernel = np.ones((3, 3), np.uint8)
binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

cv2.imshow("Binary Image", binary)
cv2.waitKey(0)

# ================================
# STEP 3: FIND CONTOURS
# ================================

contours, _ = cv2.findContours(
    binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
)

print("Total contours found:", len(contours))

# ================================
# STEP 4: SHAPE DETECTION + COUNTING
# ================================

output = img.copy()
object_count = 0

for cnt in contours:
    area = cv2.contourArea(cnt)

    # Ignore very small objects (noise)
    if area < 500:
        continue

    object_count += 1

    # Shape approximation
    epsilon = 0.02 * cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, epsilon, True)

    shape = "Unknown"

    # Identify shape by number of vertices
    if len(approx) == 3:
        shape = "Triangle"

    elif len(approx) == 4:
        x, y, w, h = cv2.boundingRect(approx)
        aspect_ratio = float(w) / h

        if 0.95 <= aspect_ratio <= 1.05:
            shape = "Square"
        else:
            shape = "Rectangle"

    elif len(approx) > 6:
        shape = "Circle"

    # Draw contour
    cv2.drawContours(output, [approx], -1, (0, 255, 0), 2)

    # Find centroid
    M = cv2.moments(cnt)
    if M["m00"] != 0:
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])

        # Label shape
        cv2.putText(
            output,
            shape,
            (cx - 30, cy),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 0, 255),
            2
        )

# ================================
# STEP 5: DISPLAY OBJECT COUNT
# ================================

cv2.putText(
    output,
    f"Objects Counted: {object_count}",
    (10, 30),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.9,
    (255, 0, 0),
    2
)

cv2.imshow("Shape Detector & Object Counter", output)
cv2.waitKey(0)
cv2.destroyAllWindows()


Total contours found: 55


### 🧠 WHY THIS PROJECT IS IMPORTANT (EXAM + INTERVIEW 🔥)
Concept	---------------------------- Used
- Thresholding	------------------------ Object separation
- Morphology	------------------------ Clean boundaries
- Contours	------------------------ Object detection
- Approximation	------------------------ Shape recognition
- Moments	------------------------ Centroid detection
- Area filtering ------------------------	Noise removal

In [ ]:
# Edge Detection using Canny
edges = cv2.Canny(blur, 50, 150)

cv2.imshow("Canny Edge Detection", edges)
cv2.waitKey(0)
cv2.destroyAllWindows()